In [ ]:
# -*- coding: utf-8 -*-
# ========== ENV / THREADS ==========
import os, re, unicodedata, warnings, shutil, time, urllib.parse
from datetime import datetime
from typing import List, Tuple, Optional

os.environ.setdefault("OMP_NUM_THREADS", "12")
os.environ.setdefault("MKL_NUM_THREADS", "12")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")
os.environ.setdefault("PYTORCH_FLASH_ATTENTION", "0")

import torch
torch.set_num_threads(12)

import pandas as pd
from docx import Document
from striprtf.striprtf import rtf_to_text

from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    pipeline
)

# LangChain – nové veci
from langchain.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceInferenceAPIEmbeddings
from langchain.embeddings import HuggingFaceEmbeddings  # fallback

# ========== CONFIG ==========
# Mód výstupu: 'title' alebo 'keyword'
MODE: str = "title"

INPUT_DIR  = "./Input"
OUTPUT_DIR = "./Output"

# Tezaurus
THESAURUS_XLSX = "./Thesaurus/SK_Local_Thesaurus.xlsx"
THESAURUS_COL  = "slovak_terms"
STOPWORDS_TXT  = "./Input/stopword_dictionary.txt"

# Modely
SUMM_MODEL_ID = "slovak-nlp/mistral-sk-7b"
GEN_MODEL_ID  = "Qwen/Qwen3-4B-Instruct-2507"

# Device map / HW
USE_CUDA   = torch.cuda.is_available()
DEVICE_MAP = {"": "cuda:0"} if USE_CUDA else None

# RAG (Chroma)
RAG_PERSIST_DIR = "./RAG_store"
RAG_COLLECTION  = "sk_legal_local"
EMB_NAME        = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
RAG_TOP_K       = 4

# Web-scraping allowlist
ALLOWED_DOMAINS = {"slov-lex.sk", "justice.gov.sk", "najpravo.sk"}

# Pipeline parametre
MAX_TEXT_CHARS  = 8000
BATCH_SIZE      = 6
GEN_ARGS = dict(max_new_tokens=72, temperature=0.35, top_p=0.9, return_full_text=False)

# Chunkovanie – použijeme aj pre DOCX/RTF, aj web
TEXT_SPLITTER = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
)

# ========== UTILS ==========
def load_stopwords(path=STOPWORDS_TXT) -> set:
    try:
        with open(path, "r", encoding="utf-8") as f:
            return {line.strip().lower() for line in f if line.strip()}
    except FileNotFoundError:
        return set()

STOPWORDS = load_stopwords()

def strip_accents(s: str) -> str:
    return ''.join(ch for ch in unicodedata.normalize('NFD', s) if not unicodedata.combining(ch))

def truncate_chars(s: str, max_chars: int) -> str:
    s = (s or "").strip()
    if len(s) <= max_chars:
        return s
    return s[: max_chars // 2] + "\n...\n" + s[- max_chars // 2 :]

def safe_first_line(s: str) -> str:
    s = (s or "").strip().strip(' "„”')
    lines = [ln for ln in s.splitlines() if ln.strip()]
    return lines[0].strip() if lines else ""

# ========== TEZAURUS ==========
def load_thesaurus_terms(xlsx_path=THESAURUS_XLSX, column=THESAURUS_COL) -> List[str]:
    if not os.path.exists(xlsx_path):
        print(f"[TH] Missing: {xlsx_path}")
        return []
    df = pd.read_excel(xlsx_path, engine="openpyxl")
    if column not in df.columns:
        raise ValueError(f'Column "{column}" not found. Columns: {list(df.columns)}')
    raw = df[column].dropna().astype(str).tolist()
    parts = []
    for cell in raw:
        parts.extend(re.split(r'[,\n;]+', cell))
    seen, terms = set(), []
    for t in parts:
        t = (t or "").strip()
        low = t.lower()
        if not t or len(low) == 1 or low in STOPWORDS:
            continue
        if t not in seen:
            seen.add(t)
            terms.append(t)
    print(f"[TH] Loaded {len(terms)} terms.")
    return terms

TERMS = load_thesaurus_terms()

def match_terms(text: str, limit=30) -> List[str]:
    if not text:
        return []
    hits, text_na = {}, strip_accents(text)
    for t in TERMS:
        rx = re.compile(r"\b" + re.escape(t) + r"\b", re.IGNORECASE)
        cnt = len(rx.findall(text))
        if cnt:
            hits[t] = cnt
    ordered = [k for k, _ in sorted(hits.items(), key=lambda kv: (-kv[1], -len(kv[0])))]
    return ordered[:limit]

# ========== DOCX/RTF SPLIT ==========
_HEADING_NAME_PREFIXES = ('heading', 'nadpis', 'po-heading-id')
_HEADING_EXACT_NAMES   = {'title', 'subtitle', 'toc heading', 'obsah', 'po-heading-id_'}
_PAGE_LINE_RE = re.compile(r'^\s*(strana|page)\s+\d+(?:\s*/\s*\d+)?\s*$', re.IGNORECASE)
_RUBRIC_RE    = re.compile(r'([.\-–—]{3,})')

def _gather_header_footer_text(doc: Document):
    lines = []
    for sec in doc.sections:
        for part in (sec.header, sec.footer):
            for p in part.paragraphs:
                t = p.text.strip()
                if t:
                    lines.append(t)
            for tb in part.tables:
                for row in tb.rows:
                    for cell in row.cells:
                        for p in cell.paragraphs:
                            t = p.text.strip()
                            if t:
                                lines.append(t)
    return set(lines)

def _looks_like_heading_style(p):
    try:
        name = (p.style.name or '').strip().lower()
    except Exception:
        name = ''
    try:
        sid = (getattr(p.style, 'style_id', '') or '').lower()
    except Exception:
        sid = ''
    if any(name.startswith(pref) for pref in _HEADING_NAME_PREFIXES):
        return True
    if name in _HEADING_EXACT_NAMES:
        return True
    if re.match(r'^heading\d+$', name) or re.match(r'^nadpis\s*\d+$', name):
        return True
    if re.match(r'^heading\d+$', sid):
        return True
    return False

def _is_header_footer_like(line: str):
    l = line.strip()
    if not l:
        return False
    if _PAGE_LINE_RE.match(l):
        return True
    if _RUBRIC_RE.search(l):
        return True
    if len(l.split()) <= 6 and l == l.upper() and any(ch.isalpha() for ch in l):
        return True
    return False

def split_docx_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        doc = Document(path)
        hf = _gather_header_footer_text(doc)
        sections, current, buf, full = [], None, [], []
        for p in doc.paragraphs:
            t = p.text.strip()
            if not t:
                continue
            if t in hf or _is_header_footer_like(t):
                continue
            if _looks_like_heading_style(p):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = t, []
            else:
                full.append(t)
                if current is not None:
                    buf.append(t)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] DOCX split failed {path}: {e}')
        return []

def _is_heading_line(l: str):
    words = l.split()
    if len(words) <= 8:
        alpha = sum(ch.isalpha() for ch in l)
        if alpha and sum(ch.isupper() for ch in l if ch.isalpha()) / alpha >= 0.7:
            return True
        if re.match(r'^\d+(\.\d+)*\s+\S', l):
            return True
    return False

def split_rtf_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        raw = rtf_to_text(open(path, 'r', encoding='utf-8').read())
        lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
        lines = [ln for ln in lines if not _is_header_footer_like(ln)]
        sections, current, buf, full = [], None, [], []
        for ln in lines:
            if _is_heading_line(ln):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = ln, []
            else:
                full.append(ln)
                if current is not None:
                    buf.append(ln)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] RTF split failed {path}: {e}')
        return []

# ========== RAG (Chroma + Embeddings) ==========
from langchain_community.vectorstores import Chroma

HF_API_KEY = os.getenv("HUGGINGFACE_API_KEY")

if HF_API_KEY:
    EMB = HuggingFaceInferenceAPIEmbeddings(
        api_key=HF_API_KEY,
        model_name=EMB_NAME,
    )
    print("[EMB] Using HuggingFaceInferenceAPIEmbeddings (remote).")
else:
    EMB = HuggingFaceEmbeddings(model_name=EMB_NAME)
    print("[EMB] Using local HuggingFaceEmbeddings (no HUGGINGFACE_API_KEY).")

def open_vectorstore():
    os.makedirs(RAG_PERSIST_DIR, exist_ok=True)
    try:
        vs = Chroma(
            collection_name=RAG_COLLECTION,
            persist_directory=RAG_PERSIST_DIR,
            embedding_function=EMB
        )
        _ = vs._collection.get(limit=1)
        print("[RAG] Ready.")
        return vs
    except Exception as e:
        print(f"[RAG] Recreate store: {e}")
        if os.path.isdir(RAG_PERSIST_DIR):
            shutil.rmtree(RAG_PERSIST_DIR, ignore_errors=True)
        os.makedirs(RAG_PERSIST_DIR, exist_ok=True)
        vs = Chroma(
            collection_name=RAG_COLLECTION,
            persist_directory=RAG_PERSIST_DIR,
            embedding_function=EMB
        )
        print("[RAG] Fresh store.")
        return vs

VS = open_vectorstore()

def rag_upsert(doc_id: str, texts: List[str], metas: List[dict]):
    """
    Vylepšený ingest:
    - Každý text rozsekáme RecursiveCharacterTextSplitrom
    - zachováme meta, doplníme index chunku
    """
    if VS is None or not texts:
        return

    chunk_texts = []
    chunk_metas = []
    for base_idx, (t, m) in enumerate(zip(texts, metas)):
        chunks = TEXT_SPLITTER.split_text(t)
        for ci, c in enumerate(chunks):
            chunk_texts.append(c)
            meta = dict(m)
            meta["chunk"] = ci
            meta["section_idx"] = base_idx
            chunk_metas.append(meta)

    ids = [f"{doc_id}_{i}" for i in range(len(chunk_texts))]
    VS.add_texts(texts=chunk_texts, metadatas=chunk_metas, ids=ids)

def rag_search(query: str, k=RAG_TOP_K) -> List[str]:
    if VS is None:
        return []
    try:
        docs = VS.similarity_search(query, k=k)
        return [d.page_content for d in docs]
    except Exception as e:
        print(f"[RAG] search failed: {e}")
        return []

# ========== WEB SCRAPING cez WebBaseLoader ==========
def _is_allowed(url: str) -> bool:
    netloc = urllib.parse.urlparse(url).netloc.lower()
    return any(netloc.endswith(d) for d in ALLOWED_DOMAINS)

def scrape_and_ingest(urls: List[str]):
    """
    Namiesto vlastného requests + trafilatura:
    - použijeme WebBaseLoader
    - rozsekáme obsah cez RecursiveCharacterTextSplitter
    - uložíme do Chroma
    """
    if VS is None:
        print("[SCRAPE] Disabled (no vector store).")
        return

    allowed = [u for u in urls if _is_allowed(u)]
    if not allowed:
        print("[SCRAPE] No allowed URLs.")
        return

    try:
        loader = WebBaseLoader(allowed)
        docs = loader.load()
    except Exception as e:
        print(f"[SCRAPE] Loader error: {e}")
        return

    if not docs:
        print("[SCRAPE] Loader returned no docs.")
        return

    split_docs = TEXT_SPLITTER.split_documents(docs)
    texts = [d.page_content for d in split_docs]
    metas = [d.metadata for d in split_docs]

    VS.add_texts(texts=texts, metadatas=metas)
    print(f"[SCRAPE] Ingested {len(split_docs)} chunks from {len(allowed)} URLs.")

# ========== PIPELINES (Mistral + Phi-3) ==========
def load_summarizer():
    tok = AutoTokenizer.from_pretrained(SUMM_MODEL_ID)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        SUMM_MODEL_ID,
        device_map=DEVICE_MAP,
        dtype=torch.float16 if USE_CUDA else torch.float32,
    )

    return pipeline(
        "text-generation",
        model=model,
        tokenizer=tok,
    )

def load_generator():
    tok = AutoTokenizer.from_pretrained(GEN_MODEL_ID)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        GEN_MODEL_ID,
        device_map=DEVICE_MAP,
        dtype=torch.float16 if USE_CUDA else torch.float32,
    )

    return pipeline(
        "text-generation",
        model=model,
        tokenizer=tok,
    )

SUMM = load_summarizer()
GEN  = load_generator()
print(f"[SUMM] {SUMM_MODEL_ID}")
print(f"[GEN ] {GEN_MODEL_ID}")

# ========== PROMPTS ==========
def prompt_summary(text: str) -> str:
    t = truncate_chars(text, MAX_TEXT_CHARS)
    return (
        "Zhrň nasledovný právny text do 1–3 viet, maximálne približne 70 slov. "
        "Buď vecný, píš po slovensky. Neuvádzaj kapitoly ani odkazy.\n\nTEXT:\n" + t
    )

def prompt_from_summary(mode: str, summary: str, terms: List[str], ctx_docs: List[str]) -> str:
    ctx = ("\n\nKONTEXT:\n" + "\n---\n".join(ctx_docs)) if ctx_docs else ""
    if mode == "title":
        return (
            f"Na základe súhrnu nižšie navrhni JEDEN presný slovenský právny nadpis (3–12 slov). "
            f"Bez úvodzoviek a bez bodky. Preferuj pojmy z tezauru: {', '.join(terms[:20])}.\n\n"
            f"SÚHRN:\n{summary}{ctx}\n\nNADPIS:"
        )
    else:
        return (
            f"Na základe súhrnu vyber JEDEN najvhodnejší slovenský právny pojem (1–4 slová). "
            f"Preferuj tezaurus: {', '.join(terms[:20])}.\n\n"
            f"SÚHRN:\n{summary}{ctx}\n\nPOJEM:"
        )

# ========== BATCH INFERENCE ==========
def summarize_batch(texts: List[str]) -> List[str]:
    outputs = []
    for i in range(0, len(texts), BATCH_SIZE):
        chunk = texts[i:i + BATCH_SIZE]
        prompts = [prompt_summary(t) for t in chunk]

        res = SUMM(
            prompts,
            batch_size=min(BATCH_SIZE, len(chunk)),
            max_new_tokens=128,
            do_sample=False,
        )

        for r in res:
            if isinstance(r, list):
                r = r[0]

            s = (
                r.get("summary_text")
                or r.get("generated_text")
                or ""
            )

            s = s.strip()
            s = re.sub(r'(?s)^.*?ZHRNUTIE:\s*', '', s)
            s = re.sub(r'\s+', ' ', s).strip()
            outputs.append(s)
    return outputs

def generate_from_summaries(mode: str, summaries: List[str], originals: List[str]):
    outputs, rag_terms_all, rag_docs_all = [], [], []
    for i, s in enumerate(summaries):
        th = match_terms(originals[i], limit=30)
        rag_terms_all.append(th)
        ctx_docs = rag_search(s, k=RAG_TOP_K)
        rag_docs_all.append(ctx_docs)
        prompt = prompt_from_summary(mode, s, th or TERMS[:40], ctx_docs)
        gen = GEN(prompt, batch_size=1, **GEN_ARGS)
        txt = gen[0]["generated_text"] if isinstance(gen, list) else str(gen)
        outputs.append(safe_first_line(txt))
    return outputs, rag_terms_all, rag_docs_all

# ========== DRIVER ==========
def process_all(input_dir=INPUT_DIR):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    files = sorted(os.listdir(input_dir), key=str.lower)
    jobs = []  # (file, header, text)

    # ingest + sekcie
    for f in files:
        p = os.path.join(input_dir, f)
        if f.lower().endswith(".docx"):
            secs = split_docx_by_headers(p)
        elif f.lower().endswith(".rtf"):
            secs = split_rtf_by_headers(p)
        else:
            continue

        if not secs:
            print(f"[WARN] No text in {f}")
            continue

        # RAG ingest (sekcie okrem __DOCUMENT__), už cez TEXT_SPLITTER v rag_upsert
        section_texts, metas = [], []
        for idx, (header, text) in enumerate(secs):
            if header == "__DOCUMENT__":
                continue
            if text.strip():
                section_texts.append(text.strip())
                metas.append({"file": f, "header": header, "idx": idx})
        if section_texts:
            rag_upsert(os.path.splitext(f)[0], section_texts, metas)

        for header, text in secs:
            if text.strip():
                jobs.append((f, header, text))

        print(f"[INFO] Queued {f} with {len(secs)} sections.")

    if not jobs:
        print("[INFO] Nothing to do.")
        return pd.DataFrame([])

    # Stage 1: summaries
    texts     = [j[2] for j in jobs]
    summaries = summarize_batch(texts)

    # Stage 2: generate (z summary + RAG + tezaurus)
    gens, rag_terms_per_section, rag_docs_per_section = generate_from_summaries(MODE, summaries, texts)

    # Compose rows
    rows = []
    for (f, header, _txt), summary, gen, th, ctx in zip(
        jobs, summaries, gens, rag_terms_per_section, rag_docs_per_section
    ):
        row = {
            "file": f,
            "header": header,
            "summary": summary,
            "suggested_title" if MODE == "title" else "top_keyword": gen,
            "method": f"SUMM+RAG+GEN ({SUMM_MODEL_ID} -> {GEN_MODEL_ID})",
            "priority_terms": "; ".join(th[:20]),
        }
        rows.append(row)

    # Save CSV
    today = datetime.today().strftime("%Y-%m-%d")
    csv_path = os.path.join(
        OUTPUT_DIR,
        f"{'titles' if MODE == 'title' else 'keywords'}_{today}.csv"
    )
    df = pd.DataFrame(rows)
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"[OK] Saved CSV: {csv_path}")

    # Simple HTML “graphic” RAG report
    html = [
        "<html><head><meta charset='utf-8'><style>",
        "body{font-family:Inter,Segoe UI,Arial,sans-serif;margin:24px;background:#0b0f13;color:#e6eef7}",
        "table{border-collapse:collapse;width:100%;}",
        "th,td{border-bottom:1px solid #253041;padding:10px;vertical-align:top}",
        "th{position:sticky;top:0;background:#121a22;text-align:left}",
        ".file{color:#9bd;} .header{color:#ffc97b}",
        ".ctx{color:#b4c6eb} .terms{color:#7fe7c4}",
        ".tag{background:#203042;border-radius:10px;padding:2px 8px;margin-right:6px;display:inline-block}",
        "</style></head><body><h2>RAG report</h2><table>",
        "<tr><th>file</th><th>header</th><th>summary</th><th>output</th><th>priority_terms</th><th>RAG ctx (top)</th></tr>",
    ]
    for r, ctx in zip(rows, rag_docs_per_section):
        terms_html = " ".join(
            f"<span class='tag'>{t}</span>"
            for t in (r["priority_terms"].split("; ") if r["priority_terms"] else [])[:12]
        )
        ctx_html = "<br/><br/>".join(
            urllib.parse.quote_plus(c[:500]).replace("+", " ")
            for c in (ctx or [])[:3]
        )
        html.append(
            f"<tr><td class='file'>{r['file']}</td>"
            f"<td class='header'>{r['header']}</td>"
            f"<td>{r['summary']}</td>"
            f"<td><b>{r['suggested_title'] if MODE=='title' else r['top_keyword']}</b></td>"
            f"<td class='terms'>{terms_html}</td>"
            f"<td class='ctx'>{ctx_html}</td></tr>"
        )
    html += ["</table></body></html>"]
    html_path = os.path.join(OUTPUT_DIR, f"rag_report_{today}.html")
    with open(html_path, "w", encoding="utf-8") as f:
        f.write("\n".join(html))
    print(f"[OK] Saved HTML: {html_path}")

    return df

# ========== MAIN ==========
if __name__ == "__main__":
    warnings.filterwarnings("ignore", message=r".*Some weights.*not of the same dtype.*")
    warnings.filterwarnings("ignore", message=r".*Tokenizer parallelism.*")

    # WebBaseLoader ingest – rovno ide do RAG cez TEXT_SPLITTER
    scrape_and_ingest([
        "https://www.slov-lex.sk/pravne-predpisy/SK/ZZ/2005/300/",
    ])

    df = process_all(INPUT_DIR)
    try:
        from IPython.display import display
        display(df.head(20))
    except Exception:
        print(df.to_string(index=False))

USER_AGENT environment variable not set, consider setting it to identify your requests.


[TH] Loaded 3000 terms.


C:\Users\nyx3t\AppData\Local\Temp\ipykernel_8156\1827601830.py:258: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  EMB = HuggingFaceEmbeddings(model_name=EMB_NAME)
c:\Users\nyx3t\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


[EMB] Using local HuggingFaceEmbeddings (no HUGGINGFACE_API_KEY).


C:\Users\nyx3t\AppData\Local\Temp\ipykernel_8156\1827601830.py:264: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vs = Chroma(


...

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Device set to use cuda:0


[SUMM] slovak-nlp/mistral-sk-7b
[GEN ] Qwen/Qwen3-4B-Instruct-2507
[SCRAPE] Ingested 1 chunks from 1 URLs.
[INFO] Queued 117694.docx with 8 sections.
[INFO] Queued 117695.docx with 2 sections.
[INFO] Queued 117696.docx with 11 sections.
[INFO] Queued 117697.docx with 15 sections.
[INFO] Queued 117698.docx with 8 sections.
[INFO] Queued 117699.docx with 18 sections.
[INFO] Queued 117700.docx with 1 sections.
[INFO] Queued 117701.docx with 6 sections.
[INFO] Queued 117702.docx with 12 sections.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[INFO] Queued 117703.docx with 10 sections.
